# 04 · The project database: every table and view with one `SELECT *`

All project data lives in one DuckDB file, `data/cropyield.duckdb` (git-ignored, built locally). Every table holds
one kind of data and has the key `obs_id` (one field in one season), so any two tables join on it. The SQL that
defines the tables and views is in [`sql/`](../sql/README.md): read `sql/schema.sql` for what every column means.

**Build it first** (from the repository root; run again after every new download or feature run):

```bash
python scripts/get_data.py                    # 1. labels check, Presto weights, Sentinel-2 chips (default: the 48-record pilot)
python scripts/features/mosaiks_features.py   # 2. MOSAIKS features from the chips -> data/features/mosaiks/
python scripts/build_database.py              # 3. data/cropyield.duckdb from 1 and 2
```

| Name | One row per | What it holds |
|---|---|---|
| `labels` | field-season (all cohorts) | yield, unit, location, polygon, dates, season window |
| `downloads` | field-season with a chip | download status, pixel counts, search window |
| `scenes` | accepted satellite scene | date, satellite, tile, cloud numbers |
| `coverage` | field-season × month | no scene / cloudy / clear / download error |
| `monthly_bands` | field-season × month | median reflectance of the field per band (B02 … B12) |
| `vegetation_indices` | field-season × month | NDVI, GCVI, MTCI, NDVI705, NDVI740 |
| `presto_embeddings` | field-season with a usable month | `embedding`: the 128 frozen Presto numbers as one array |
| `mosaiks_monthly` | field-season × month | `chip`, `field`: MOSAIKS vectors (K numbers each) |
| `mosaiks_season` | field-season | season mean of the MOSAIKS vectors |
| `build_info` | build | when, from which code and which folders the file was built |
| `season_indices` (view) | field-season | season max and mean of each index |
| **`master`** (view) | field-season (all cohorts) | labels + status + season indices + Presto + MOSAIKS |
| **`monthly`** (view) | field-season × month | the month-by-month sequence for GRU-type models |

Vectors are stored as **arrays**: one value per row. `cropyield.data.database.wide` spreads one into numbered
columns for scikit-learn (last section).

In [1]:
import pandas as pd

from cropyield.data.database import connect, run_query, wide

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 170)
con = connect()   # read-only: several notebooks can read at the same time. Raises if the file is missing.

## What is inside

`run_query(con, name)` runs the saved file `sql/queries/<name>.sql`. This one counts every table and view.

In [2]:
run_query(con, "table_overview")

,name,kind,rows
0,labels,table,11364
1,downloads,table,48
2,scenes,table,335
3,coverage,table,420
4,monthly_bands,table,420
5,vegetation_indices,table,420
6,presto_embeddings,table,48
7,mosaiks_monthly,table,335
8,mosaiks_season,table,48
9,build_info,table,1


## The master view: one row per field

In [3]:
master = con.sql("SELECT * FROM master").df()
print(master.shape, "| rows with imagery:", int(master["has_imagery"].sum()))
master[master["has_imagery"]].head()

(11364, 37) | rows with imagery: 48


,obs_id,cohort,dataset,source_id,source_row,country,year,yield_kg_ha,source_yield,source_yield_units,yield_field,latitude,...,gcvi_season_max,gcvi_season_mean,mtci_season_max,mtci_season_mean,ndvi705_season_max,ndvi705_season_mean,ndvi740_season_max,ndvi740_season_mean,presto_embedding,mosaiks_chip,mosaiks_field,mosaiks_months_used
11316,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,malawi_2023,malawi_2023,506ae8b2-d085-4eeb-b807-765e03925c71,1,malawi,2023,2950.0,2.95,t/ha,cc_ey_t,-13.317068,...,2.870482,1.886488,3.154965,2.195516,0.408726,0.256032,0.085123,0.060906,"[-0.87177867, -4.672151, 2.1013925, -0.1017125...","[2.1101022, 1.6449106, 0.47028247, 1.8732961, ...","[2.5186625, 1.4577835, 0.31989312, 1.7050442, ...",5
11317,malawi_2023:626290f6-99b2-4a44-b233-8df0e13d3e63,malawi_2023,malawi_2023,626290f6-99b2-4a44-b233-8df0e13d3e63,2,malawi,2023,2300.0,2.30,t/ha,cc_ey_t,-13.322024,...,3.385366,2.487342,3.031475,2.245230,0.463996,0.300572,0.076019,0.063838,"[-0.6986065, -4.595742, 1.9818386, -0.21318507...","[0.6132526, 1.4513252, 1.1122215, 1.1919034, 1...","[0.27337545, 1.7042334, 1.2021481, 1.0589778, ...",4
11318,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,malawi_2023,10738537-aded-4abe-bc18-e3d8ce52f7f7,3,malawi,2023,2620.0,2.62,t/ha,cc_ey_t,-13.291493,...,2.630058,1.849790,2.682441,2.034604,0.330685,0.237776,0.082909,0.057956,"[-0.9493931, -4.732916, 2.094909, -0.3450331, ...","[1.1774728, 2.104361, 1.3814399, 1.5121517, 1....","[1.875757, 2.6309707, 0.07524905, 2.123213, 0....",5
11319,malawi_2023:1d60c29e-ee3a-4639-9e07-0239523dfc62,malawi_2023,malawi_2023,1d60c29e-ee3a-4639-9e07-0239523dfc62,4,malawi,2023,3310.0,3.31,t/ha,cc_ey_t,-13.289999,...,3.316011,2.137250,3.490281,2.229132,0.483500,0.276881,0.078453,0.058948,"[-0.9957593, -4.741645, 2.1107435, -0.4867943,...","[0.8962175, 2.1824622, 1.1032529, 1.6514323, 0...","[0.23271218, 2.4462695, 0.92539215, 1.5426482,...",5
11320,malawi_2023:6fa8a978-4710-469b-b593-e100746035dd,malawi_2023,malawi_2023,6fa8a978-4710-469b-b593-e100746035dd,5,malawi,2023,4510.0,4.51,t/ha,cc_ey_t,-13.302169,...,4.101404,2.334993,3.324852,2.394438,0.511549,0.307511,0.084483,0.056045,"[-1.3451, -5.0080495, 1.8070883, -0.3263331, -...","[2.8837972, 2.4895775, 0.9314534, 1.7594013, 0...","[1.6268976, 2.5263724, 2.6324368, 1.6687996, 1...",5


Only fields with downloaded chips have features; the others keep their label row with empty features
(`has_imagery = false`). For modelling, filter on it:

In [4]:
con.sql("""
    SELECT cohort,
           count(*)                                                 AS fields,
           count(*) FILTER (WHERE has_imagery)                      AS with_imagery,
           count(*) FILTER (WHERE presto_embedding IS NOT NULL)     AS with_presto,
           count(*) FILTER (WHERE mosaiks_chip IS NOT NULL)         AS with_mosaiks
    FROM master
    GROUP BY cohort
    ORDER BY cohort
""").df()

,cohort,fields,with_imagery,with_presto,with_mosaiks
0,kenya_oaf_2019_2020,3977,12,12,12
1,malawi_2022,529,0,0,0
2,malawi_2023,260,12,12,12
3,zimbabwe_2024,4332,16,16,16
4,zimbabwe_lacuna_2022_2023,2266,8,8,8


## The monthly view: one row per field and month

The sequence a GRU reads: nine rows per field (the season window), `NULL` where the month has no clear scene.

In [5]:
monthly = con.sql("SELECT * FROM monthly ORDER BY obs_id, month").df()
print(monthly.shape)
monthly.head(9)

(420, 22)


,obs_id,cohort,month,coverage_state,b02,b03,b04,b05,b06,b07,b08,b8a,b11,b12,ndvi,gcvi,mtci,ndvi705,ndvi740,mosaiks_chip,mosaiks_field,yield_kg_ha
0,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,2022-10,no_scene,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2620.0
1,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,2022-11,no_scene,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2620.0
2,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,2022-12,clear,0.0639,0.09485,0.1424,0.169188,0.192622,0.207919,0.1979,0.217141,0.323175,0.289347,0.163091,1.086452,1.071862,0.078217,0.013516,"[1.1059208, 4.955574, 0.41166243, 3.4271173, 0...","[1.2940627, 7.076677, 0.0, 4.602618, 0.0, 8.66...",2620.0
3,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,2023-01,cloudy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2620.0
4,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,2023-02,cloudy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2620.0
5,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,2023-03,clear,0.1271,0.15190,0.1294,0.193325,0.331594,0.376487,0.3648,0.395406,0.267803,0.165050,0.476325,1.401580,2.682441,0.307234,0.047683,"[3.826641, 0.015497149, 4.6832743, 0.25177148,...","[6.502249, 0.0, 0.15277809, 0.2751501, 0.0, 0....",2620.0
6,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,2023-04,clear,0.0416,0.06920,0.0690,0.126350,0.219119,0.248688,0.2512,0.274306,0.241787,0.155119,0.569019,2.630058,2.176983,0.330685,0.068212,"[0.38151127, 0.13780847, 1.5419959, 0.3285668,...","[0.7296162, 0.41366938, 0.22346719, 1.3105046,...",2620.0
7,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,2023-05,clear,0.0446,0.06930,0.0772,0.130163,0.203094,0.226653,0.2372,0.251303,0.241859,0.159663,0.508906,2.422799,2.021005,0.291367,0.077462,"[0.24843708, 0.35633475, 0.27003348, 0.4911405...","[0.5463624, 0.6468048, 0.0, 0.95292896, 0.2686...",2620.0
8,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,2023-06,clear,0.0585,0.08375,0.1258,0.157159,0.192072,0.210725,0.2268,0.242066,0.308469,0.217375,0.286444,1.708060,2.220727,0.181375,0.082909,"[0.3248538, 5.0565915, 0.00023377164, 3.062162...","[0.3064946, 5.017702, 0.0, 3.474863, 0.7288048...",2620.0


## Every table on its own

In [6]:
labels = con.sql("SELECT * FROM labels").df()
labels.head()

,obs_id,cohort,dataset,source_id,source_row,country,year,yield_kg_ha,source_yield,source_yield_units,yield_field,latitude,longitude,geometry_type,field_area_ha,label_date,season_start,season_end,intercropping,field_polygon
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,malawi_2022,malawi_2022,3ee4d3a6-3a6e-4dae-9046-66009b51d884,1,malawi,2022,3731.0,3.731,t/ha,AVG_YLD,-15.052553,35.204831,Polygon,0.133246,2022-04-30,2021-10-01,2022-06-30,NaN,"{""type"": ""Polygon"", ""coordinates"": [[[35.20461..."
1,malawi_2022:4f780999-86d8-42b6-bc2a-3f45a6c56a63,malawi_2022,malawi_2022,4f780999-86d8-42b6-bc2a-3f45a6c56a63,2,malawi,2022,2488.0,2.488,t/ha,AVG_YLD,-15.046088,34.905687,Polygon,0.312441,2022-05-02,2021-10-01,2022-06-30,NaN,"{""type"": ""Polygon"", ""coordinates"": [[[34.90594..."
2,malawi_2022:9d9a8fb6-2b21-4d32-b9de-43f0d00af118,malawi_2022,malawi_2022,9d9a8fb6-2b21-4d32-b9de-43f0d00af118,3,malawi,2022,1224.0,1.224,t/ha,AVG_YLD,-15.091802,34.975304,Polygon,0.620347,2022-05-02,2021-10-01,2022-06-30,NaN,"{""type"": ""Polygon"", ""coordinates"": [[[34.97540..."
3,malawi_2022:b5dc6d2d-bc97-4f19-a3d6-9986325aeaf2,malawi_2022,malawi_2022,b5dc6d2d-bc97-4f19-a3d6-9986325aeaf2,4,malawi,2022,2033.0,2.033,t/ha,AVG_YLD,-15.044796,34.889381,Polygon,0.387880,2022-05-02,2021-10-01,2022-06-30,NaN,"{""type"": ""Polygon"", ""coordinates"": [[[34.88952..."
4,malawi_2022:0f15bad6-fd33-4d62-b60c-a9cac3671e62,malawi_2022,malawi_2022,0f15bad6-fd33-4d62-b60c-a9cac3671e62,5,malawi,2022,645.0,0.645,t/ha,AVG_YLD,-15.094286,34.979696,Polygon,0.331923,2022-05-02,2021-10-01,2022-06-30,NaN,"{""type"": ""Polygon"", ""coordinates"": [[[34.98011..."


In [7]:
downloads = con.sql("SELECT * FROM downloads").df()
downloads.head()

,obs_id,record_dir,status,aoi_type,aoi_pixels,field_pixels,aoi_inside_chip,catalog_scenes,accepted_months,crs,search_start,search_end
0,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,data/imagery/sentinel2/run-517bfcf0a69c6cac/re...,complete,field_polygon,109,79,True,50,5,EPSG:32736,2022-10-01,2023-06-30
1,malawi_2023:626290f6-99b2-4a44-b233-8df0e13d3e63,data/imagery/sentinel2/run-517bfcf0a69c6cac/re...,complete,field_polygon,73,52,True,50,4,EPSG:32736,2022-10-01,2023-06-30
2,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,data/imagery/sentinel2/run-517bfcf0a69c6cac/re...,complete,field_polygon,147,118,True,50,5,EPSG:32736,2022-10-01,2023-06-30
3,malawi_2023:1d60c29e-ee3a-4639-9e07-0239523dfc62,data/imagery/sentinel2/run-517bfcf0a69c6cac/re...,complete,field_polygon,112,84,True,50,5,EPSG:32736,2022-10-01,2023-06-30
4,malawi_2023:6fa8a978-4710-469b-b593-e100746035dd,data/imagery/sentinel2/run-517bfcf0a69c6cac/re...,complete,field_polygon,88,64,True,50,5,EPSG:32736,2022-10-01,2023-06-30


In [8]:
scenes = con.sql("SELECT * FROM scenes").df()
scenes.head()

,obs_id,item_id,acquired,month,satellite,tile,tile_cloud_cover,clear_fraction_aoi,field_valid_pixels
0,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,S2B_T36LWL_20230113T080204_L2A,2023-01-13,2023-01,S2B,36LWL,68.141800,1.0,79
1,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,S2B_T36LWL_20230307T081156_L2A,2023-03-07,2023-03,S2B,36LWL,25.330079,1.0,79
2,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,S2B_T36LWL_20230426T081153_L2A,2023-04-26,2023-04,S2B,36LWL,3.369666,1.0,79
3,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,S2B_T36LWL_20230506T081154_L2A,2023-05-06,2023-05,S2B,36LWL,0.088551,1.0,79
4,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,S2A_T36LWL_20230630T080627_L2A,2023-06-30,2023-06,S2A,36LWL,0.004809,1.0,79


In [9]:
coverage = con.sql("SELECT * FROM coverage").df()
coverage.head(9)

,obs_id,month,state,item_id
0,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2022-10,no_scene,NaN
1,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2022-11,no_scene,NaN
2,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2022-12,cloudy,NaN
3,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-01,clear,S2B_T36LWL_20230113T080204_L2A
4,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-02,cloudy,NaN
5,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-03,clear,S2B_T36LWL_20230307T081156_L2A
6,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-04,clear,S2B_T36LWL_20230426T081153_L2A
7,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-05,clear,S2B_T36LWL_20230506T081154_L2A
8,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-06,clear,S2A_T36LWL_20230630T080627_L2A


In [10]:
monthly_bands = con.sql("SELECT * FROM monthly_bands").df()
monthly_bands.head(9)

,obs_id,month,b02,b03,b04,b05,b06,b07,b08,b8a,b11,b12,field_pixels,missing
0,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2022-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,True
1,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2022-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,True
2,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2022-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,True
3,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-01,0.1140,0.1490,0.1798,0.219650,0.278031,0.313119,0.2836,0.333100,0.415538,0.389688,79,False
4,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,True
5,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-03,0.0421,0.0664,0.0606,0.107869,0.223694,0.264962,0.2570,0.279756,0.206975,0.128506,79,False
6,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-04,0.0900,0.1178,0.1180,0.173306,0.263988,0.290381,0.2986,0.314694,0.272781,0.189075,79,False
7,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-05,0.0496,0.0752,0.0862,0.141100,0.213813,0.244631,0.2536,0.269281,0.254569,0.178200,79,False
8,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-06,0.0604,0.0857,0.1190,0.159231,0.201400,0.222831,0.2358,0.254631,0.285700,0.208538,79,False


In [11]:
vegetation_indices = con.sql("SELECT * FROM vegetation_indices").df()
vegetation_indices.head(9)

,obs_id,month,ndvi,gcvi,mtci,ndvi705,ndvi740
0,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2022-10,NaN,NaN,NaN,NaN,NaN
1,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2022-11,NaN,NaN,NaN,NaN,NaN
2,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2022-12,NaN,NaN,NaN,NaN,NaN
3,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-01,0.223997,0.903356,1.604768,0.127074,0.009915
4,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-02,NaN,NaN,NaN,NaN,NaN
5,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-03,0.618388,2.870482,3.154965,0.408726,0.069288
6,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-04,0.433509,1.534805,2.265453,0.265506,0.061524
7,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-05,0.492643,2.372340,2.049180,0.285027,0.085123
8,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-06,0.329200,1.751459,1.903216,0.193830,0.078683


In [12]:
presto_embeddings = con.sql("SELECT * FROM presto_embeddings").df()
print(presto_embeddings.shape, "| one embedding is an array of", len(presto_embeddings["embedding"].iloc[0]), "numbers")
presto_embeddings.head()

(48, 2) | one embedding is an array of 128 numbers


,obs_id,embedding
0,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,"[-0.87177867, -4.672151, 2.1013925, -0.1017125..."
1,malawi_2023:626290f6-99b2-4a44-b233-8df0e13d3e63,"[-0.6986065, -4.595742, 1.9818386, -0.21318507..."
2,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,"[-0.9493931, -4.732916, 2.094909, -0.3450331, ..."
3,malawi_2023:1d60c29e-ee3a-4639-9e07-0239523dfc62,"[-0.9957593, -4.741645, 2.1107435, -0.4867943,..."
4,malawi_2023:6fa8a978-4710-469b-b593-e100746035dd,"[-1.3451, -5.0080495, 1.8070883, -0.3263331, -..."


In [13]:
mosaiks_monthly = con.sql("SELECT * FROM mosaiks_monthly").df()
print(mosaiks_monthly.shape, "| this table stays empty until scripts/features/mosaiks_features.py has run")
mosaiks_monthly.head()

(335, 4) | this table stays empty until scripts/features/mosaiks_features.py has run


,obs_id,month,chip,field
0,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-01,"[4.6942515, 0.8359695, 1.4088441, 2.0557923, 0...","[5.418028, 3.2894256, 0.037633922, 4.299364, 0..."
1,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-03,"[2.031045, 0.6865135, 0.7004669, 1.1891172, 0....","[0.93917656, 0.0, 1.52876, 0.0002873125, 0.156..."
2,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-04,"[1.7239368, 0.7007714, 0.20012699, 1.2588038, ...","[5.712377, 0.0, 0.030752322, 1.278423, 0.38904..."
3,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-05,"[1.0664808, 1.6805778, 0.04197419, 1.6664139, ...","[0.28945783, 0.49447206, 0.002319505, 0.866892..."
4,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,2023-06,"[1.0347962, 4.320721, 0.0, 3.1963532, 0.128562...","[0.2342722, 3.5050197, 0.0, 2.0802538, 0.53630..."


In [14]:
mosaiks_season = con.sql("SELECT * FROM mosaiks_season").df()
mosaiks_season.head()

,obs_id,months_used,chip,field
0,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,5,"[2.1101022, 1.6449106, 0.47028247, 1.8732961, ...","[2.5186625, 1.4577835, 0.31989312, 1.7050442, ..."
1,malawi_2023:626290f6-99b2-4a44-b233-8df0e13d3e63,4,"[0.6132526, 1.4513252, 1.1122215, 1.1919034, 1...","[0.27337545, 1.7042334, 1.2021481, 1.0589778, ..."
2,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,5,"[1.1774728, 2.104361, 1.3814399, 1.5121517, 1....","[1.875757, 2.6309707, 0.07524905, 2.123213, 0...."
3,malawi_2023:1d60c29e-ee3a-4639-9e07-0239523dfc62,5,"[0.8962175, 2.1824622, 1.1032529, 1.6514323, 0...","[0.23271218, 2.4462695, 0.92539215, 1.5426482,..."
4,malawi_2023:6fa8a978-4710-469b-b593-e100746035dd,5,"[2.8837972, 2.4895775, 0.9314534, 1.7594013, 0...","[1.6268976, 2.5263724, 2.6324368, 1.6687996, 1..."


In [15]:
con.sql("SELECT * FROM build_info").df().T

,0
built_at,2026-10-29 06:22:21.910199
git_commit,6a2b9c502693a1ad9966e327d345abaad77a4f48
presto_commit,11e207a668a34336ced1d8e492a1bd5849b96c4a
presto_weights_sha256,559c9d76f525bdf2dd3774da8b1e7f1510769b73baacac...
presto_embeddings_built,True
imagery_dir,data/imagery/sentinel2
n_labels,11364
n_downloads,48
n_embeddings,48
mosaiks_features,1024


## The saved queries in `sql/queries/`

Each file starts with a comment that says what it returns. Open one to see plain SQL you can copy.

**`coverage_by_cohort.sql`**: how many field-months of each cohort are clear, cloudy or without a scene.

In [16]:
run_query(con, "coverage_by_cohort")

,cohort,fields_with_imagery,field_months,clear,cloudy,no_scene,download_error,clear_percent
0,kenya_oaf_2019_2020,12,96,90,0,6,0,93.8
1,malawi_2023,12,108,65,19,24,0,60.2
2,zimbabwe_2024,16,144,142,2,0,0,98.6
3,zimbabwe_lacuna_2022_2023,8,72,38,6,28,0,52.8


**`modelling_table.sql`**: one cohort, one row per field with imagery: label plus every feature.
Change the cohort in the file's `WHERE` line.

In [17]:
modelling = run_query(con, "modelling_table")
modelling.head()

,obs_id,cohort,yield_kg_ha,months_with_data,ndvi_season_max,ndvi_season_mean,gcvi_season_max,gcvi_season_mean,mtci_season_max,mtci_season_mean,ndvi705_season_max,ndvi705_season_mean,ndvi740_season_max,ndvi740_season_mean,presto_embedding,mosaiks_chip,mosaiks_field,mosaiks_months_used
0,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,malawi_2023,2620.0,5,0.569019,0.400757,2.630058,1.849790,2.682441,2.034604,0.330685,0.237776,0.082909,0.057956,"[-0.9493931, -4.732916, 2.094909, -0.3450331, ...","[1.1774728, 2.104361, 1.3814399, 1.5121517, 1....","[1.875757, 2.6309707, 0.07524905, 2.123213, 0....",5
1,malawi_2023:1d60c29e-ee3a-4639-9e07-0239523dfc62,malawi_2023,3310.0,5,0.722051,0.457717,3.316011,2.137250,3.490281,2.229132,0.483500,0.276881,0.078453,0.058948,"[-0.9957593, -4.741645, 2.1107435, -0.4867943,...","[0.8962175, 2.1824622, 1.1032529, 1.6514323, 0...","[0.23271218, 2.4462695, 0.92539215, 1.5426482,...",5
2,malawi_2023:4591d3d5-8258-4a9e-a8db-ad1024acb672,malawi_2023,780.0,7,0.683375,0.462061,2.518672,1.805996,2.828016,2.034621,0.381190,0.265395,0.084367,0.047047,"[-1.1697931, -4.9489927, 2.0409489, -0.2971767...","[2.7199082, 1.6770895, 1.4517311, 1.5266296, 1...","[4.154716, 1.5710529, 1.158848, 1.7314652, 0.4...",7
3,malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,malawi_2023,2950.0,5,0.618388,0.419547,2.870482,1.886488,3.154965,2.195516,0.408726,0.256032,0.085123,0.060906,"[-0.87177867, -4.672151, 2.1013925, -0.1017125...","[2.1101022, 1.6449106, 0.47028247, 1.8732961, ...","[2.5186625, 1.4577835, 0.31989312, 1.7050442, ...",5
4,malawi_2023:626290f6-99b2-4a44-b233-8df0e13d3e63,malawi_2023,2300.0,4,0.728569,0.499391,3.385366,2.487342,3.031475,2.245230,0.463996,0.300572,0.076019,0.063838,"[-0.6986065, -4.595742, 1.9818386, -0.21318507...","[0.6132526, 1.4513252, 1.1122215, 1.1919034, 1...","[0.27337545, 1.7042334, 1.2021481, 1.0589778, ...",4


**`monthly_series.sql`**: the month-by-month rows of that cohort, in time order.

In [18]:
run_query(con, "monthly_series").head(18)

,obs_id,month,coverage_state,b02,b03,b04,b05,b06,b07,b08,b8a,b11,b12,ndvi,gcvi,mtci,ndvi705,ndvi740,mosaiks_chip,mosaiks_field,yield_kg_ha
0,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,2022-10,no_scene,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2620.0
1,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,2022-11,no_scene,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2620.0
2,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,2022-12,clear,0.0639,0.09485,0.1424,0.169188,0.192622,0.207919,0.1979,0.217141,0.323175,0.289347,0.163091,1.086452,1.071862,0.078217,0.013516,"[1.1059208, 4.955574, 0.41166243, 3.4271173, 0...","[1.2940627, 7.076677, 0.0, 4.602618, 0.0, 8.66...",2620.0
3,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,2023-01,cloudy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2620.0
4,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,2023-02,cloudy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2620.0
5,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,2023-03,clear,0.1271,0.15190,0.1294,0.193325,0.331594,0.376487,0.3648,0.395406,0.267803,0.165050,0.476325,1.401580,2.682441,0.307234,0.047683,"[3.826641, 0.015497149, 4.6832743, 0.25177148,...","[6.502249, 0.0, 0.15277809, 0.2751501, 0.0, 0....",2620.0
6,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,2023-04,clear,0.0416,0.06920,0.0690,0.126350,0.219119,0.248688,0.2512,0.274306,0.241787,0.155119,0.569019,2.630058,2.176983,0.330685,0.068212,"[0.38151127, 0.13780847, 1.5419959, 0.3285668,...","[0.7296162, 0.41366938, 0.22346719, 1.3105046,...",2620.0
7,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,2023-05,clear,0.0446,0.06930,0.0772,0.130163,0.203094,0.226653,0.2372,0.251303,0.241859,0.159663,0.508906,2.422799,2.021005,0.291367,0.077462,"[0.24843708, 0.35633475, 0.27003348, 0.4911405...","[0.5463624, 0.6468048, 0.0, 0.95292896, 0.2686...",2620.0
8,malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,2023-06,clear,0.0585,0.08375,0.1258,0.157159,0.192072,0.210725,0.2268,0.242066,0.308469,0.217375,0.286444,1.708060,2.220727,0.181375,0.082909,"[0.3248538, 5.0565915, 0.00023377164, 3.062162...","[0.3064946, 5.017702, 0.0, 3.474863, 0.7288048...",2620.0
9,malawi_2023:1d60c29e-ee3a-4639-9e07-0239523dfc62,2022-10,no_scene,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,3310.0


## From `master` to X and y for scikit-learn

The modelling table has one row per field. `y` is the yield. Three feature matrices come from it:
the season indices (10 plain columns), the Presto embedding (one array per row) and the MOSAIKS chip
vector (one array per row). `wide` turns an array column into numbered columns; a row without a
vector becomes a row of NaN, so the rows always line up with `y`.

In [19]:
rows = modelling.reset_index(drop=True)
y = rows["yield_kg_ha"].to_numpy()

index_columns = [column for column in rows.columns if column.endswith("_season_max") or column.endswith("_season_mean")]
X_indices = rows[index_columns]                                 # 10 columns
X_presto = wide(rows, "presto_embedding", "presto")             # presto_000 ... presto_127
X_mosaiks = (wide(rows, "mosaiks_chip", "mosaiks")              # mosaiks_0000 ... (K columns)
             if rows["mosaiks_chip"].notna().any() else None)   # None until the MOSAIKS features are built

print("y:", y.shape, "| X_indices:", X_indices.shape, "| X_presto:", X_presto.shape,
      "| X_mosaiks:", None if X_mosaiks is None else X_mosaiks.shape)
X_presto.iloc[:5, :6]

y: (12,) | X_indices: (12, 10) | X_presto: (12, 128) | X_mosaiks: (12, 1024)


,presto_000,presto_001,presto_002,presto_003,presto_004,presto_005
0,-0.949393,-4.732916,2.094909,-0.345033,-0.450171,1.598379
1,-0.995759,-4.741645,2.110744,-0.486794,-0.492089,1.038312
2,-1.169793,-4.948993,2.040949,-0.297177,-0.408481,1.533949
3,-0.871779,-4.672151,2.101393,-0.101713,-0.317664,1.611898
4,-0.698606,-4.595742,1.981839,-0.213185,-0.214862,1.372112


Close the connection when you are done. On Windows an open connection locks the file, and
`scripts/build_database.py` cannot rebuild it while any notebook still holds it.

In [20]:
con.close()